# Qwen3-0.6B × 커스텀 Triton AttentionBackend (multi-seq paged)

vLLM 의 공식 `AttentionBackend` + `vllm.general_plugins` entry point.

**이 프로젝트의 학습 포인트**: `vllm_paged` 의 두 커널 (prefill·decode paged) 을 유지하되,
이제 두 커널 모두 **여러 시퀀스를 한 번의 launch 로 처리**한다.

- Prefill kernel: flat-packed Q `[total_q_tokens, Hq, D]` + `query_start_loc` 로 per-seq 경계 구분. Causal mask 는 절대 위치 `q_abs = seq_len - q_len + local` 기반
- Decode kernel: Q `[num_decode_seqs, Hq, D]` — 각 seq 당 1 토큰을 한 번에 배치 처리
- **Backend 가 혼합 배치 분할**: prefill 시퀀스들은 prefill 커널 1회, decode 시퀀스들은 decode 커널 1회 (vLLM v0 의 실제 스타일)

다음 단계 (`vllm_unified`): 이 두 커널을 **하나의 커널**로 통합 + find\_seq\_idx / chunked prefill.


## 준비물 & 설치

- CUDA GPU 필수
- Python >= 3.10
- vLLM **0.19.1 정확히** 권장 (내부 API drift)

```bash
# 노트북 디렉토리에서
pip install -e .
```

`pyproject.toml` 의 entry point (`vllm.general_plugins = triton_attention_backend:register`) 가
모든 vLLM 프로세스(메인·엔진 코어·워커) 에서 자동으로 `register()` 를 호출해 준다.
즉 **노트북 안에서 `register()` 를 수동 호출하지 않는다**.

절차:
1. cell 3 — 환경 확인
2. cell 8 — 커널 smoke test
3. cell 10 — LLM 생성 (plugin 이 이미 CUSTOM 슬롯 점유)
4. cell 11~13 — generate + 실행 증거 확인


In [ ]:
import torch, triton, vllm

print('cuda:', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('vllm:', vllm.__version__)
print('triton:', triton.__version__)

if not torch.cuda.is_available():
    raise SystemExit('이 노트북은 CUDA GPU가 필요합니다.')

# 이 노트북은 vLLM 0.19.x 기준으로 작성됨. 다른 버전에서는 내부 API drift 가능.
assert vllm.__version__.startswith('0.19'), (
    f'vLLM 0.19.x 권장 (현재: {vllm.__version__}). '
    '다른 버전은 AttentionBackend 내부 API 가 다를 수 있음.'
)


## Qwen3-0.6B 구조 요약

| 항목 | 값 |
|---|---|
| hidden_size | 1024 |
| Q heads | 16 |
| KV heads | 8 (GQA 2:1) |
| head_dim | 128 |
| layers | 28 |

In [ ]:
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained('Qwen/Qwen3-0.6B')
hd = cfg.head_dim if hasattr(cfg, 'head_dim') else cfg.hidden_size // cfg.num_attention_heads
print('hidden:', cfg.hidden_size)
print('Q heads:', cfg.num_attention_heads, '/ KV heads:', cfg.num_key_value_heads)
print('head_dim:', hd)
print('layers:', cfg.num_hidden_layers)

## Triton 커널 역할 (multi-seq)

`triton_attn.py`:
- `_fwd_kernel_prefill_multiseq` — flat-packed Q + `query_start_loc`. grid `(num_seqs * Hq, max_q_blocks)`. 각 프로그램이 `batch_idx` 로 자기 seq 의 `(q_start, q_len, s_len, block_table row)` 를 읽어 처리
- `_fwd_kernel_decode_multiseq` — Q `[num_decode_seqs, Hq, D]`. grid `(num_decode_seqs * Hq,)`. 각 프로그램이 한 seq 의 1 토큰 + 전체 KV 에 attention

Backend (`MyTritonImpl.forward`):
```python
q_lens = query_start_loc[1:] - query_start_loc[:-1]
is_prefill = q_lens == seq_lens
is_decode  = q_lens == 1

if is_prefill.any(): run prefill kernel on prefill subgroup
if is_decode.any():  run decode kernel on decode subgroup
```

**한 forward 안에 prefill + decode 가 섞이면** backend 가 두 그룹을 분할해 각각의 커널을 **한 번씩만** 호출.
이게 진짜 batching — 커널 호출 수가 시퀀스 수에 비례하지 않음.


In [ ]:
from triton_attn import (
    triton_attention_prefill_multiseq,
    triton_attention_decode_multiseq,
    _pack_to_paged_multiseq,
)
print('prefill wrapper:'); print(triton_attention_prefill_multiseq.__doc__)
print()
print('decode  wrapper:'); print(triton_attention_decode_multiseq.__doc__)


In [ ]:
# Smoke test (multi-seq):
#   1) 3개 시퀀스, 각기 다른 길이
#   2) 각 seq 별 SDPA 로 reference 계산
#   3) 공유 paged cache 구성 후 우리 커널 호출
#   4) 각 seq 별로 오차 비교
import torch
import torch.nn.functional as F

Hq, Hkv, D = 16, 8, 128
dtype = torch.float16

q_lens = [32, 64, 128]
q_per_seq, k_per_seq, v_per_seq, ref_per_seq = [], [], [], []
for S in q_lens:
    q_s = torch.randn(Hq, S, D, dtype=dtype, device='cuda')
    k_s = torch.randn(Hkv, S, D, dtype=dtype, device='cuda')
    v_s = torch.randn(Hkv, S, D, dtype=dtype, device='cuda')
    q_per_seq.append(q_s); k_per_seq.append(k_s); v_per_seq.append(v_s)
    k_rep = k_s.repeat_interleave(Hq // Hkv, dim=0)
    v_rep = v_s.repeat_interleave(Hq // Hkv, dim=0)
    ref = F.scaled_dot_product_attention(
        q_s.unsqueeze(0), k_rep.unsqueeze(0), v_rep.unsqueeze(0), is_causal=True
    ).squeeze(0)
    ref_per_seq.append(ref)

# Flat-pack Q
q_flat = torch.cat([q.transpose(0, 1) for q in q_per_seq], dim=0)
cumulative = torch.cumsum(torch.tensor(q_lens, dtype=torch.int32), 0)
query_start_loc = torch.cat([torch.tensor([0], dtype=torch.int32), cumulative]).to('cuda')
seq_lens = torch.tensor(q_lens, dtype=torch.int32, device='cuda')

key_cache, value_cache, block_table = _pack_to_paged_multiseq(k_per_seq, v_per_seq, block_size=16)
ours = triton_attention_prefill_multiseq(
    q_flat, key_cache, value_cache, block_table, seq_lens, query_start_loc
)

max_err = 0.0
for i, S in enumerate(q_lens):
    start = int(query_start_loc[i].item())
    ours_seq = ours[start:start + S].transpose(0, 1)
    err = (ours_seq.float() - ref_per_seq[i].float()).abs().max().item()
    max_err = max(max_err, err)

print(f'multi-seq prefill 3 seqs q_lens={q_lens} max_abs_err = {max_err:.6f}  →  {"PASS" if max_err < 1e-2 else "FAIL"}')


## Plugin entry point + multi-seq 분할 dispatch

`pyproject.toml`:
```toml
[project.entry-points."vllm.general_plugins"]
my_multiseq_backend = "triton_attention_backend:register"
```

**Backend 의 multi-seq 핵심 로직** (`MyTritonImpl.forward`):

```python
q_lens = query_start_loc[1:] - query_start_loc[:-1]
is_prefill = q_lens == seq_lens     # 전체 q 를 prefill
is_decode  = q_lens == 1            # 1 토큰만

if is_prefill.any():
    # prefill seqs 의 q 토큰들을 flat 하게 모아 한 번의 커널 호출
    o_p = triton_attention_prefill_multiseq(q_prefill, ..., sl_prefill, bt_prefill, qsl_prefill)

if is_decode.any():
    # decode seqs 의 마지막 q 토큰을 batch 축에 쌓아 한 번의 커널 호출
    o_d = triton_attention_decode_multiseq(q_decode, ..., sl_decode, bt_decode)
```

**이게 vLLM v0 실제 스타일**: prefill/decode 커널은 분리돼 있지만 각각은 multi-seq batched.
`max_num_seqs > 1` 로 continuous batching 을 실감할 수 있음.

실행 증거: 엔진 코어 stderr 의 `fired (prefill multiseq) / (decode multiseq)` 로그 + num\_seqs 표시.


In [ ]:
# entry point 등록 상태 확인
from importlib.metadata import entry_points

eps = list(entry_points(group='vllm.general_plugins'))
mine = [e for e in eps if e.name == 'my_multiseq_backend']
assert mine, (
    '`my_multiseq_backend` entry point 가 보이지 않는다. '
    '노트북 디렉토리에서 `pip install -e .` 을 실행했는지 확인하라.'
)
print('plugin registered:', mine[0].name, '->', mine[0].value)


In [ ]:
from vllm import LLM, SamplingParams
from vllm.v1.attention.backends.registry import AttentionBackendEnum

# max_num_seqs > 1 로 설정 — 여러 요청이 한 배치에 섞여 들어오는 상황을 실감
llm = LLM(
    model='Qwen/Qwen3-0.6B',
    dtype='float16',
    attention_backend=AttentionBackendEnum.CUSTOM,
    enforce_eager=True,
    max_num_seqs=4,
    max_model_len=2048,
)


In [ ]:
# 4개 프롬프트를 동시에 제출 — vLLM 이 batch 로 묶어 단일 forward 로 처리한다
prompts = [
    'The capital of France is',
    'The largest planet in our solar system is',
    'Shakespeare wrote the play',
    'Python was created by',
]
out = llm.generate(prompts, SamplingParams(temperature=0, max_tokens=16))
for i, o in enumerate(out):
    print(f'[{i}]', o.outputs[0].text)


## 검증: 엔진 코어 stderr 의 fired 로그

`MyTritonImpl.forward` 가 호출될 때마다 prefill / decode 경로별로 로그 한 줄씩 찍는다
(각 경로 첫 1회만). 위 cell 11 출력 **바로 위** 에 다음과 같이 나왔어야:

```
(EngineCore pid=XXXXX) MyTritonImpl.forward fired (prefill multiseq) num_seqs=4 tokens=28
(EngineCore pid=XXXXX) MyTritonImpl.forward fired (decode multiseq) num_seqs=4
```

`num_seqs=4` 가 바로 **multi-seq batching** 의 증거 — 한 forward 에 4개 시퀀스가 함께 처리됨.


In [ ]:
# 노트북 프로세스에서 확인 가능한 것:
# (1) plugin 이 등록돼 있고 (2) CUSTOM 슬롯에 우리 백엔드 경로가 꽂혔는가
from vllm.v1.attention.backends.registry import AttentionBackendEnum

path = AttentionBackendEnum.CUSTOM.get_path()
print('CUSTOM slot ->', path)
assert 'MyTritonBackend' in path
print('OK — plugin 자동 등록 확인')
print()
print('실제 multi-seq 실행 증거는 위 cell 11 출력 바로 위의 `num_seqs=...` 로그 확인')


## 다음 단계 — `vllm_unified`

- **단일 커널**로 prefill + decode 혼합 배치 처리 — backend 의 분할 dispatch 제거
- **`find_seq_idx`** 로 flat grid 지원 (vLLM v1 실제 방식)
- **chunked prefill** — q\_len > 1 이면서 < s\_len 인 경우도 지원
- **split-k (stage1/stage2)** — 긴 컨텍스트 decode 의 parallelism
